# 🐍 Ekans – User Guide

Ekans provides a dynamic, reloadable, flattened view of a Python package.

Instead of writing:

In [8]:
from arbok_driver import Ekans
from arbok_driver.examples import sub_sequences, configurations

In [9]:
ekans_sub_sequences = Ekans(sub_sequences)
ekans_configurations = Ekans(configurations)

In [11]:
ekans_sub_sequences.reload_modules()

you can write:

In [2]:
#ekans.device.configs.master_configs.master_33333333_conf

It does this by:

discovering modules dynamically
importing/reloading them
attaching their public attributes into a structured namespace

## ⚙️ 1. Basic Usage
Initialize Ekans
import my_package
ekans = Ekans(my_package)
Access attributes
ekans.analysis.DecayFit
ekans.device.configs.master_configs.my_config
Reload everything
ekans.reload_modules()

This will:

reload all modules
rebuild the attribute tree
reflect code changes without restarting Python

⚠️ Note: Python’s reload system has limitations (existing objects may still reference old definitions). See Python docs
.


## 📁 2. Required Folder Structure

Ekans relies on Python’s package system.

Minimal valid structure

```
arbok_driver
├── ...
├── examples
│   ├── configurations
│   │   ├── __init__.py
│   │   ├── hardware_configs
│   │   │   ├── divider_config.py
│   │   │   ├── __init__.py
│   │   │   ├── opx1000_config.py
│   │   └── sequence_configs
│   │       ├── __init__.py
│   │       ├── coulomb_peaks_config.py
│   │       ├── device_config.py
│   │       ├── parity_init_conf.py
│   │       ├── parity_readout_conf.py
│   │       ├── square_pulse_confs.py
│   │       └── stability_map_config.py
│   ├── experiments
│   │   ├── __init__.py
│   │   └── square_pulse_experiments.py
│   ├── readout_classes
│   │   ├── __init__.py
│   │   ├── dc_average.py
│   │   ├── dc_chopped_readout.py
│   │   ├── difference.py
│   │   └── threshold.py
│   └── sub_sequences
│       ├── __init__.py
│       ├── coulomb_peaks.py
│       ├── parity_initialization.py
│       ├── parity_readout.py
│       ├── square_pulse_legacy.py
│       ├── square_pulse.py
│       ├── square_pulse_scalable.py
│       └── stability_map.py
├── ...
```

### Rules
Every folder must be a package
Include an __init__.py in every directory. Otherwise, Python may treat it as a namespace package, which can break module discovery.
Only .py files are considered modules
Other files (e.g., .txt, .json) are ignored. They do not break Ekans, but are not imported or exposed.
Modules must be importable
Each .py file should not raise errors at import time (unless intentional) and should use valid imports.

## 🧠 3. How Ekans Builds the Namespace
Uses pkgutil.walk_packages to discover module names.
Imports or reloads each module using importlib.
Builds a tree of _NamespaceNode objects.
Attaches attributes from modules, flattening the hierarchy one level.
Example transformation

File structure:

```python
analysis/
    decay_fit.py   (contains class DecayFit)
```
Normal Python:
```
from analysis.decay_fit import DecayFit
```
With Ekans:
```
ekans.analysis.DecayFit
```

## 🎯 4. Controlling What Gets Exposed

Ekans follows this rule:

If __all__ exists → only those names are exposed
Otherwise → all public attributes are exposed
Option A — Expose everything (default)
```
# decay_fit.py
class DecayFit:
    ...
```
Available as:
```
ekans.analysis.DecayFit
Option B — Restrict using __all__
# decay_fit.py
__all__ = ["DecayFit"]
```
Only DecayFit is exposed.

Option C — Restrict at package level
```
# master_configs/__init__.py
__all__ = ["rf2v_opx1000_swapped"]
```
Only the explicitly listed submodules are exposed.

⚠️ If you define __all__, the names must exist in the module namespace:

from . import rf2v_opx1000_swapped

## 🔄 5. Reload Behavior

Ekans calls importlib.reload(module).

Works well for:
function changes
constants
simple class definitions
Limitations:
existing instances keep old class definitions
from x import y does not update automatically

## 🧪 6. Best Practices

Always import modules, not objects
```
import my_module
my_module.my_function()
```
This ensures reload works correctly.

Use __all__ for clean APIs
Keeps the namespace predictable and prevents name collisions.
Keep modules small
Prefer multiple smaller modules over one giant file.

Avoid side effects at import
Bad:
```
connect_to_database()  # runs on import
```
Good:
```
def connect():
    ...
```
Debug missing modules
```
print(name in sys.modules)
```
If False → module was never imported.


## 🐛 7. Common Pitfalls
Problem	Likely Cause
Module not showing up	Not imported or missing __init__.py
Code not executing	Module never imported
Attribute missing	Filtered by __all__ or not defined
Reload not updating behavior	Python reload limitation (object references persist)
🏁 Summary

Ekans provides:

Benefits:

Dynamic module discovery
Hot-reloading
Flattened API
Reduced import boilerplate

Requirements:

Proper package structure (__init__.py)
Modules must be importable
Understand Python reload limitations

Key idea:

Ekans separates what gets imported (everything) from what gets exposed (controlled via __all__).